# Where do AI scribes fail? Analysis

Run top to bottom after Day 3. Every table and figure used in the write-up is produced here
and saved to `outputs/tables/` and `outputs/figures/`.

**Set `DEMO = True`** to run on fake annotations (`python scripts/make_demo_annotations.py`)
before you've labeled anything. Never report DEMO numbers.

In [ ]:
DEMO = False

import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from src.common import ANN, OUT, load_config, load_traps, taxonomy_codes
from src import stats_utils as su

cfg = load_config()
CODES = taxonomy_codes()
ann_dir = ANN / "demo" if DEMO else ANN
(OUT / "tables").mkdir(parents=True, exist_ok=True); (OUT / "figures").mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3); pd.set_option("display.max_colwidth", 120)
if DEMO: print("*** DEMO MODE: fake annotations, do not report these numbers ***")

## 1. Data

In [ ]:
h_notes, h_errs, h_traps = su.load_human("main", ann_dir)
H = su.note_level(h_notes, h_errs).merge(h_notes[["system", "encounter_id", "quality", "minutes"]], on=["system", "encounter_id"])
H["source"] = np.where(H.encounter_id.str.startswith("ED"), "synthetic_ed", "aci_bench")
SYSTEMS = [s["name"] for s in cfg["systems"] if s["name"] in set(H.system)]
print(f"{len(H)} reviewed notes, {len(h_errs)} human-logged errors, {H.minutes.sum()/60:.1f} h annotation")
H.groupby(["system", "source"]).size().unstack(fill_value=0)

## 2. Error rates by system (physician labels)
Note-level rates with Wilson 95% CIs; errors per note with bootstrap CIs.

In [ ]:
rows = []
for s in SYSTEMS:
    d = H[H.system == s]
    n = len(d)
    for col, label in [("any_error", "≥1 error"), ("any_critical", "≥1 critical error")]:
        p, lo, hi = su.wilson(int(d[col].sum()), n)
        rows.append({"system": s, "metric": label, "estimate": p, "ci_low": lo, "ci_high": hi, "n": n})
    m, lo, hi = su.bootstrap_mean_ci(d.n_errors)
    rows.append({"system": s, "metric": "errors per note", "estimate": m, "ci_low": lo, "ci_high": hi, "n": n})
    m, lo, hi = su.bootstrap_mean_ci(d.quality)
    rows.append({"system": s, "metric": "quality (1-5)", "estimate": m, "ci_low": lo, "ci_high": hi, "n": n})
rates = pd.DataFrame(rows)
rates.to_csv(OUT / "tables/error_rates.csv", index=False)
rates.pivot(index="metric", columns="system", values="estimate")

In [ ]:
# Error profile: share of notes with at least one error of each category, by system
prof = H.groupby("system")[[f"cat_{c}" for c in CODES]].mean().T
prof.index = [i.replace("cat_", "") for i in prof.index]
prof = prof[SYSTEMS]
prof.to_csv(OUT / "tables/error_profile.csv")

# Severity mix
sev = h_errs.groupby(["system", "severity"]).size().unstack(fill_value=0).reindex(SYSTEMS)
display(prof.round(2)); display(sev)

fig, ax = plt.subplots(figsize=(8, 4.2))
prof.plot.barh(ax=ax, width=0.8)
ax.invert_yaxis(); ax.set_xlabel("Share of notes with ≥1 error of this type"); ax.set_ylabel("")
ax.legend(title="", frameon=False); ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Error profile by system (physician labels)", loc="left")
plt.tight_layout(); plt.savefig(OUT / "figures/error_profile.png", dpi=200); plt.show()

## 3. Ship decision: does the new prompt reduce critical errors?
Pre-specified in `config.yaml` before looking at results: endpoint, reference, candidate and
hypothesis (superiority or non-inferiority). Same model, same encounters, so the test is paired.

In [ ]:
dc = cfg["decision"]
ref, new, col = dc["reference_system"], dc["candidate_system"], dc["primary_endpoint"]
pw = su.paired(H, ref, new, col)
mc = su.mcnemar_exact(pw[ref], pw[new])
dec = su.decide(pw[ref], pw[new], dc.get("hypothesis", "superiority"),
                margin=dc.get("noninferiority_margin", 0.10), alpha=dc["alpha"])
print(f"Paired encounters: {len(pw)}")
print(f"{col}: {ref} = {pw[ref].mean():.3f}, {new} = {pw[new].mean():.3f}")
print(f"McNemar exact: only {ref} had the error = {mc['ref_only']}, only {new} had it = {mc['new_only']}, p = {mc['p_value']:.3f}")
print(f"Difference ({new} - {ref}) = {dec['diff_new_minus_ref']:+.3f}, "
      f"{dec['ci_level']:.0%} CI [{dec['ci_low']:+.3f}, {dec['ci_high']:+.3f}]; {dec['hypothesis']} threshold {dec['threshold']:+.2f}")
label = {"superiority": ("SUPERIOR: the new prompt reduces critical errors", "Superiority NOT shown"),
         "noninferiority": ("NON-INFERIOR within the margin", "Non-inferiority NOT shown")}[dec["hypothesis"]]
print("=> " + (label[0] if dec["met"] else label[1]))
pd.DataFrame([{**mc, **dec, "ref": ref, "new": new, "endpoint": col, "n_pairs": len(pw)}]).to_csv(OUT / "tables/ship_decision.csv", index=False)

### 3b. What did the prompt change? Category-level trade-offs
Paired differences in the share of notes with each error type (candidate minus reference).
Look for a trade-off: rules like "never add anything not stated" may cut hallucinations but
make the model more conservative, which can raise omissions. Exploratory, not adjusted for multiplicity.

In [ ]:
rows = []
for c in CODES:
    pw_c = su.paired(H, ref, new, f"cat_{c}")
    if pw_c[[ref, new]].values.sum() == 0:
        continue
    d, lo, hi = su.paired_diff_ci(pw_c[ref], pw_c[new])
    rows.append({"category": c, ref: pw_c[ref].mean(), new: pw_c[new].mean(), "diff": d, "ci_low": lo, "ci_high": hi,
                 "mcnemar_p": su.mcnemar_exact(pw_c[ref], pw_c[new])["p_value"]})
CAT = pd.DataFrame(rows).sort_values("diff")
CAT.to_csv(OUT / "tables/prompt_effect_by_category.csv", index=False)
display(CAT)

fig, ax = plt.subplots(figsize=(7, 0.45 * len(CAT) + 1.2))
ax.axvline(0, color="gray", lw=0.8)
ax.errorbar(CAT["diff"], CAT.category, xerr=[CAT["diff"] - CAT.ci_low, CAT.ci_high - CAT["diff"]], fmt="o", capsize=3)
ax.set_xlabel(f"Change in share of notes with this error ({new} − {ref}), 95% CI")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Which errors did the prompt change fix, and which did it cause?", loc="left")
plt.tight_layout(); plt.savefig(OUT / "figures/prompt_effect_by_category.png", dpi=200); plt.show()

In [ ]:
# Note length and cost by system (longer notes take longer to review before signing)
from src.common import read_jsonl
N = pd.DataFrame(read_jsonl(OUT / "notes.jsonl"))
N["words"] = N.note.str.split().str.len()
N.groupby("system")[["words", "input_tokens", "output_tokens", "latency_s"]].mean().reindex(SYSTEMS).round(1)

In [ ]:
# Optional verify arm: how much did the self-audit pass change the draft?
if "draft_note" in N and N.draft_note.notna().any():
    import difflib
    V = N[N.draft_note.notna()].copy()
    V["similarity_to_draft"] = [difflib.SequenceMatcher(None, a, b).ratio() for a, b in zip(V.draft_note, V.note)]
    V["words_added"] = V.words - V.draft_note.str.split().str.len()
    display(V.groupby("system")[["similarity_to_draft", "words_added"]].describe().round(2))
else:
    print("No verify arm in this run.")

In [ ]:
# All pairwise comparisons (exploratory; not multiplicity-adjusted)
out = []
for i, a in enumerate(SYSTEMS):
    for b in SYSTEMS[i + 1:]:
        for c in ["any_critical", "any_error"]:
            pw2 = su.paired(H, a, b, c)
            d, lo, hi = su.paired_diff_ci(pw2[a], pw2[b])
            out.append({"a": a, "b": b, "endpoint": c, "n": len(pw2), "diff_b_minus_a": d, "ci_low": lo, "ci_high": hi,
                        "mcnemar_p": su.mcnemar_exact(pw2[a], pw2[b])["p_value"]})
pd.DataFrame(out)

## 4. Synthetic ED trap checklist
Each trap is one pre-specified clinical fact; pass rate per system and per trap category.

In [ ]:
traps = load_traps(cfg)
T = h_traps[h_traps.result.isin(["pass", "fail"])].merge(traps[["trap_id", "category", "check"]], on="trap_id")
if T.empty:
    print("No trap results yet.")
else:
    T["passed"] = (T.result == "pass").astype(int)
    by_sys = T.groupby("system").passed.agg(["mean", "sum", "count"]).reindex(SYSTEMS)
    by_sys[["ci_low", "ci_high"]] = [su.wilson(int(r["sum"]), int(r["count"]))[1:] for _, r in by_sys.iterrows()]
    display(by_sys)
    display(T.pivot_table(index="category", columns="system", values="passed", aggfunc="mean")[SYSTEMS].round(2))
    fails = T[T.result == "fail"].sort_values(["trap_id", "system"])[["trap_id", "system", "check"]]
    fails.to_csv(OUT / "tables/trap_failures.csv", index=False)
    display(fails)

## 5. Do automated metrics track clinical errors?
ACI-Bench encounters only (they have reference notes). If ROUGE barely correlates with
physician-judged errors, that's a headline finding.

In [ ]:
am_path = OUT / "auto_metrics.csv"
if am_path.exists():
    am = pd.read_csv(am_path).merge(H, on=["system", "encounter_id"])
    metric_cols = [c for c in ["rouge1", "rouge2", "rougeL", "bertscore_f1"] if c in am]
    corr = []
    for m in metric_cols:
        for target in ["n_errors", "n_critical", "quality"]:
            rho, p = spearmanr(am[m], am[target])
            corr.append({"metric": m, "vs": target, "spearman_rho": rho, "p": p, "n": len(am)})
    corr = pd.DataFrame(corr); corr.to_csv(OUT / "tables/metric_correlations.csv", index=False)
    display(corr)
    # Do metric-based and clinician-based system rankings agree?
    rank = pd.DataFrame({"mean_rougeL": am.groupby("system").rougeL.mean(),
                         "critical_rate": am.groupby("system").any_critical.mean(),
                         "quality": am.groupby("system").quality.mean()}).reindex(SYSTEMS)
    display(rank)
    fig, ax = plt.subplots(figsize=(6, 4))
    for s in SYSTEMS:
        d = am[am.system == s]
        ax.scatter(d.rougeL, d.n_critical + np.random.default_rng(0).uniform(-.12, .12, len(d)), label=s, alpha=.75)
    ax.set_xlabel("ROUGE-L vs reference note"); ax.set_ylabel("Critical errors (physician, jittered)")
    ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("Overlap metrics vs clinically critical errors", loc="left")
    plt.tight_layout(); plt.savefig(OUT / "figures/rouge_vs_critical.png", dpi=200); plt.show()
else:
    print("Run `python -m src.auto_metrics` first.")

## 6. Can the LLM judge be trusted?
Physician labels are the reference standard. Report kappa, and sensitivity / PPV for
**critical** errors specifically: that's what decides whether a judge could replace human review.

In [ ]:
j_notes, j_errs = su.load_judge()
print(f"Judge status counts: {j_notes.status.value_counts().to_dict()}")
J = su.note_level(j_notes[j_notes.status == "ok"], j_errs)
M = H.merge(J, on=["system", "encounter_id"], suffixes=("_h", "_j"))
print(f"{len(M)} notes labeled by both")

ag = [{"level": "note", "target": t, **su.agreement(M[f"{t}_h"], M[f"{t}_j"])} for t in ["any_error", "any_critical"]]
ag += [{"level": "category", "target": c, **su.agreement(M[f"cat_{c}_h"], M[f"cat_{c}_j"])} for c in CODES]
AG = pd.DataFrame(ag)
AG.to_csv(OUT / "tables/judge_agreement.csv", index=False)
AG[["level", "target", "n", "prevalence", "kappa", "sensitivity", "sens_lo", "sens_hi", "ppv", "ppv_lo", "ppv_hi", "specificity"]]

In [ ]:
# Does the judge rank systems the same way the physician does? (Judge bias by system)
cmp = pd.DataFrame({
    "human_critical_rate": M.groupby("system").any_critical_h.mean(),
    "judge_critical_rate": M.groupby("system").any_critical_j.mean(),
    "human_errors_per_note": M.groupby("system").n_errors_h.mean(),
    "judge_errors_per_note": M.groupby("system").n_errors_j.mean(),
}).reindex(SYSTEMS)
cmp.to_csv(OUT / "tables/judge_vs_human_by_system.csv"); cmp

In [ ]:
# Per-category sensitivity with CIs (the chart for the write-up)
cat = AG[(AG.level == "category") & ((AG.tp + AG.fn) > 0)].sort_values("sensitivity")
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(cat.sensitivity, cat.target, xerr=[cat.sensitivity - cat.sens_lo, cat.sens_hi - cat.sensitivity], fmt="o", capsize=3)
for _, r in cat.iterrows():
    ax.annotate(f"n={r.tp + r.fn}", (r.sens_hi, r.target), xytext=(6, -3), textcoords="offset points", fontsize=8, color="gray")
ax.set_xlim(0, 1.05); ax.set_xlabel("Judge sensitivity vs physician (95% CI)")
ax.spines[["top", "right"]].set_visible(False); ax.set_title("Which errors does the LLM judge catch?", loc="left")
plt.tight_layout(); plt.savefig(OUT / "figures/judge_sensitivity_by_category.png", dpi=200); plt.show()

## 7. Intra-rater reliability
Re-labeled subset (Day 3), same rater, blinded to first-pass labels.

In [ ]:
try:
    r_notes, r_errs, _ = su.load_human("relabel", ann_dir)
    R = su.note_level(r_notes, r_errs)
    RR = H.merge(R, on=["system", "encounter_id"], suffixes=("_1", "_2"))
    irr = [{"target": t, **su.agreement(RR[f"{t}_1"], RR[f"{t}_2"])} for t in ["any_error", "any_critical"]]
    irr += [{"target": c, **su.agreement(RR[f"cat_{c}_1"], RR[f"cat_{c}_2"])} for c in CODES if RR[[f"cat_{c}_1", f"cat_{c}_2"]].values.any()]
    IRR = pd.DataFrame(irr)[["target", "n", "prevalence", "kappa", "tp", "fp", "fn", "tn"]]
    IRR.to_csv(OUT / "tables/intra_rater.csv", index=False); display(IRR)
except FileNotFoundError as e:
    print("No re-label set yet:", e)

## 8. Failure gallery
Critical errors with evidence: pick 3 to 5 for the write-up and add your clinical commentary.

In [ ]:
gal = h_errs[h_errs.severity >= 3].copy()
jflag = j_errs.assign(j=1).groupby(["system", "encounter_id", "category"]).j.max().reset_index()
gal = gal.merge(jflag, how="left", on=["system", "encounter_id", "category"]).fillna({"j": 0})
gal["judge_caught_same_category"] = gal.j.astype(int)
cols = ["system", "encounter_id", "category", "note_text", "transcript_evidence", "comment", "judge_caught_same_category"]
gal[cols].to_csv(OUT / "tables/critical_error_gallery.csv", index=False)
gal[cols].sort_values(["category", "system"]).head(40)

## 9. Notes for the write-up
- Report n for every estimate; with ~35 notes per system, CIs are wide. Say so.
- The primary endpoint and margin were pre-specified. Everything else is exploratory.
- The judge's failure modes (section 6) and the metric/clinical disagreement (section 5) are
  usually the most interesting findings for a documentation-AI team.